## Анализ данных о Нобелевских лауреатах

### **Контекст проекта**
В этом задании мы будем работать с датасетом о лауреатах Нобелевской премии. Ваша задача — распарсить и проанализировать эти данные, ответив на ключевые вопросы о закономерностях вручения премий, демографических характеристиках лауреатов и исторических тенденциях.

### **Этап 1: Подготовка данных**

**Задача:** На этом этапе вам надо собрать структурированный список словарей с информацией о лауреатах и призах. Приведем начало нужного нам датасета.

``` {code:python}
[{'id': '745',
  'name': 'A. Michael Spence',
  'gender': 'male',
  'birth_year': 1943,
  'country_birth': 'USA',
  'country_now': 'USA',
  'prizes_relevant': [{'prize_amount': 10000000,
    'prize_amount_adjusted': 15547541,
    'award_year': 2001,
    'category_en': 'Economic Sciences',
    'prize_status': 'received'}],
  'type_': 'person'},
 {'id': '102',
  'name': 'Aage N. Bohr',
  'gender': 'male',
  'birth_year': 1922,
  'country_birth': 'Denmark',
  'country_now': 'Denmark',
  'prizes_relevant': [{'prize_amount': 630000,
    'prize_amount_adjusted': 4304697,
    'award_year': 1975,
    'category_en': 'Physics',
    'prize_status': 'received'}],
  'type_': 'person'}]
  ```

Поскольку информация о лауреатах-людях и лауреатах-организациях структурирована по-разному, а также из-за того, что часть полей в json могут отсутстввовать, необходимо реализовать функции для процессинга словарей, которые будут достаточно обобщаемыми для обоих типов данных.

**Что нужно сделать:**

На входе информация о конкретном лауреате представляет из себя кучу вложенных друг в друга словарей. Из этой структуры нам надо получить плоский словарь вида {atribute: value}, где atribute --- это один из небольшого множества интересных нам атрибутов. При этом в исходном файле 
1. пути к атрибутам могут иметь разную длину
2. атрибуты могут отсутствовать
3. атрибуты могут нуждаться в дополнительной обработке (например, поле id в исходных словарях записано как строка.)

Поскольку наивная реализация такого обработчика оказывается довольно громоздкой, вам нужно создать файл json_dict_processing.py, в котором реализован высокоуровневый функционал для решения задачи "обработать вложенный json-словарь"

Главное требование к интерфейсу: 
по (вложенному) словарю о лауреате и конфигу (см ниже) надо получить плоский словарь вида {atribute: value}.

Достичь этого можно, например, таким набором функций

* `extract_nested_value(obj, keys)` - извлекает значение из вложенных структур данных по цепочке ключей

* `process_dictionary_with_config(dictionary, config)` - обрабатывает один словарь согласно словару-конфигу (формат конфига см ниже.)

* `process_list_of_dicts_with_config(list_of_dicts, config)` - обрабатывает список словарей, эта функция нужна чтобы обработать список призов.

* `create_processor(config, list_processor=False)` - создает готовый процессор с предзагруженной конфигурацией. В случае, если list_processor == True, внутрь процессора подается process_list_of_dicts_with_config, иначе process_dictionary_with_config.

**Замечание насчет последней функции.**

Планируемое применение этой функции примерно такое:
`laureate_processor = create_processor(PERSON_CONFIG, list_processor=False)`

То есть, create_processor возвращает функцию, которая реализует функционал process_dictionary_with_config но с уже заполненным конфигом. Это повышает читаемость кода и снижает риск перепутать конфиги из-за дублирования кода.


Помимо этого нужно создать файлы-конфиги для данных по призам, лауреатам-людям, и лауреатам-организациям.

КОНФИГ для функций process_dictionary_with_config и process_list_of_dicts_with_config имеет следующий вид
```python
  {'атрибут_который_берем_сырым': ['путь', 'к', 'данным'],  # или
  'атрибут_который_нужно_обработать': (['путь', 'к', 'данным'], функция_обработки)}
  ```
В случае, если атрибут можно взять в итоговый датасет сырым, в values() словаря-конфига будет список path_list.
Если же атрибут еще надо обработать, для него в поле config[attribute] будет tuple вида (path_list, function).

Примеры атрибутов, которые можно взять сырыми: имя, размер приза. Примеры атрибутов, которые нужно обработать: год вручения приза (в сырых данных есть только полная дата, записанная как строка)

После того, как будет создан core.py, создайте конфиги laureates_configs.py и prizes_configs.py

Постарайтесь провалидировать код с помощью assert-ов, вызываемых в ноутбуке или в .py-файлах.

пример использования: 

assert 1 == True

**Валидация**

В вашем случае assert должен взять простой словарик с одним вложенным полем и какой-нибудь конфиг-файл.
Также убедитесь, что при обработке данных собственно по лауреатам в них нет странностей (смотрите ниже) 

**Атрибуты для ПРИЗА (CONFIG_PRIZE, prizes_configs.py)**

- `prize_amount` - сумма приза → `prizeAmount`
- `prize_amount_adjusted` - скорректированная сумма приза → `prizeAmountAdjusted`
- `award_year` - год вручения приза (преобразуется в целое число) → `awardYear`
- `category_en` - категория на английском → `category.en`
- `prize_status` - статус приза → `prizeStatus`

**Атрибуты для ЧЕЛОВЕКА-ЛАУРЕАТА (CONFIG_PERSON, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - имя лауреата → `knownName.en`
- `gender` - пол → `gender`
- `birth_year` - год рождения (извлекается из даты) → `birth.date`
- `country_birth` - страна рождения → `birth.place.country.en`
- `country_now` - текущая страна → `birth.place.countryNow.en`
- `prizes_relevant` - список призов лауреата → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Атрибуты для ОРГАНИЗАЦИИ-ЛАУРЕАТА (CONFIG_ORG, laureates_configs.py)**

- `id` - уникальный идентификатор → `id`
- `name` - название организации → `orgName.en`
- `founded_year` - год основания (извлекается из даты) → `founded.date`
- `country_founded` - страна основания → `founded.place.country.en`
- `country_now` - текущая страна → `founded.place.countryNow.en`
- `prizes_relevant` - список призов организации → `nobelPrizes` (обрабатывается процессором призов, который вы задаете в конфиге для призов)

**Вспомогательные функции:**

- `process_year(year_string)` - извлекает год из строки даты (формат "YYYY-MM-DD"). Функция находится в файле laureates_configs.py
- `prize_processor()` - создает процессор для обработки списков призов. Функция находится в файле prizes_configs.py
- `person_processor()` - создает процессор для данных о людях-лауреатах. Функция находится в файле laureates_configs.py
- `org_processor()` - создает процессор для данных об организациях-лауреатах. Функция находится в файле laureates_configs.py

### Импорт функций

In [1]:
import json
from json_dict_processing import (
    extract_nested_value,
    process_dictionary_with_config,
    process_list_of_dicts_with_config,
    create_processor
)
from laureates_configs import process_orgs, process_persons
from aggregations import calculate_stats
from dataset_helpers import (
    group_by_attributes,
    calculate_missing,
    split_laureates,
    find_laureate_by_id,
    get_ids_info,
    find_id_gaps
)
import re
from collections import Counter
from stopwords import STOPWORDS

### Проверка функций

Проверим на простых примерах работу функций обработки словарей, конфигов, группировки и расчёта статистик.

In [2]:
# Проверка обработки вложенных словарей
test_data = {'prize': {'year': '2020', 'category': 'Physics'}}
test_config = {
    'year': (['prize', 'year'], int),
    'category': ['prize', 'category']
}
expected = {'year': 2020, 'category': 'Physics'}

assert extract_nested_value(
    test_data, ['prize', 'category']
) == 'Physics'

assert extract_nested_value(
    test_data, ['prize', 'missing']
) is None

assert process_dictionary_with_config(
    test_data, test_config
) == expected

assert process_list_of_dicts_with_config(
    [test_data, test_data], test_config
) == [expected, expected]

assert create_processor(test_config)(test_data) == expected


# Проверка обработки лауреатов
test_person = {
    'id': '1',
    'knownName': {'en': 'Anna'},
    'gender': 'female',
    'birth': {'date': '1990-05-10'},
    'nobelPrizes': []
}

test_org = {
    'id': '2',
    'orgName': {'en': 'Test organization'},
    'founded': {'date': '1950-01-01'},
    'nobelPrizes': []
}

assert process_persons(test_person)['birth_year'] == 1990
assert process_orgs(test_org)['founded_year'] == 1950


# Проверка вспомогательных функций
test_laureates = [
    {'id': '1', 'type_': 'person'},
    {'id': '3', 'type_': 'org'}
]

persons_test, orgs_test = split_laureates(test_laureates)

assert len(persons_test) == 1
assert len(orgs_test) == 1

assert calculate_missing(
    [{'age': None}, {'age': 25}]
) == ({'age': 1}, {'age': 50.0})

found_org = find_laureate_by_id(test_laureates, 3)
assert found_org['type_'] == 'org'

assert find_laureate_by_id(test_laureates, 5) is None

assert get_ids_info(test_laureates) == ([1, 3], 0)

assert find_id_gaps([1, 2, 5]) == [(2, 5)]

assert group_by_attributes(
    [{'gender': 'female'}, {'gender': 'male'}],
    ['gender']
) == {
    ('female',): [{'gender': 'female'}],
    ('male',): [{'gender': 'male'}]
}


# Проверка расчета статистик
assert calculate_stats([10, 20, 30]) == {
    'count': 3,
    'mean': 20,
    'median': 20,
    'max': 30
}

### Загружаем данные о лауреатах из JSON-файла

In [3]:
with open('laureates.json', 'r', encoding='utf-8') as file:
    laureates = json.load(file)
print('Кол-во считанных значений', len(laureates['laureates']))

Кол-во считанных значений 1028


### Преобразуем исходные данные в список словарей. Для людей и организаций используем разные конфиги.

In [4]:
def get_laureate_data(laureate):
    """
    Определяет тип лауреата, обрабатывает его данные
    и добавляет поле type_.
    """
    if 'knownName' in laureate:
        result = process_persons(laureate)
        result['type_'] = 'person'
    elif 'orgName' in laureate:
        result = process_orgs(laureate)
        result['type_'] = 'org'
    else:
        return None
    return result


laureates_data = []

for laureate in laureates['laureates']:
    laureates_data.append(get_laureate_data(laureate))
print('Кол-во считанных значений', len(laureates_data))
print(laureates_data[0])

Кол-во считанных значений 1028
{'id': '745', 'name': 'A. Michael Spence', 'gender': 'male', 'birth_year': 1943, 'country_birth': 'USA', 'country_now': 'USA', 'prizes_relevant': [{'prize_amount': 10000000, 'prize_amount_adjusted': 15547541, 'award_year': 2001, 'category_en': 'Economic Sciences', 'prize_status': 'received', 'motivation_en': 'for their analyses of markets with asymmetric information'}], 'type_': 'person'}




### **Этап 2: EDA лауреатов**

EDA или Exploratory Data Analysis - первый шаг при работе с любыми данными.

Чтобы начать работать с этой секцией, проведите проведите чисто технический анализ собранных данных.

1. Сколько всего записей?
2. Все ли значения полей заполнены?
3. Если нет, сколько пропусков в каждом из полей? Выведите ответ на этот вопрос в формате `{'field_name': missing_values}`. А какая доля пропусков?
5. Каковы максимальные и минимальные значения поля id и каким годам награждения они соответствуют? Есть ли записи с повторяющимися id? Есть ли пары айди, которые в отсортированном массиве айдишников различаются более, чем на один? 

При реализации ответов на эти вопросы, пользуйтесь теми же принципами, что описаны выше. Можете начать с плохого хакерского кода, однако в таком случае в маркдауне пропишите, как будете его обобщать и разносить по функциям и модулям. (об этом ниже)

В основном EDA пользуйтесь аккуратным кодом, разбитым на функции и файлы.

### Первичная проверка данных. Посчитаем количество записей, пропуски и проверим id лауреатов

In [5]:
# 1. Сколько всего записей?
print('Всего записей:', len(laureates_data))
persons, orgs = split_laureates(laureates_data)
person_missings, person_share = calculate_missing(persons)
org_missings, org_share = calculate_missing(orgs)

# 2. Все ли значения полей заполнены?
print('\nВсе ли значения полей заполнены?')
if sum(person_missings.values()) + sum(org_missings.values()) == 0:
    print('Да')
else:
    print('Нет')

# 3. Количество и доля пропусков по каждому полю
print('\nПропуски у людей:')
for key, value in person_missings.items():
    print(f'{key}: {value} ({person_share[key]}%)')

print('\nПропуски у организаций:')
for key, value in org_missings.items():
    print(f'{key}: {value} ({org_share[key]}%)')

# 4. Минимальный и максимальный id и годы награждения
ids, duplicates = get_ids_info(laureates_data)
print('\nМинимальный id:', min(ids))
print('Максимальный id:', max(ids))
for id_value in (min(ids), max(ids)):
    laureate = find_laureate_by_id(laureates_data, id_value)
    print('\nid:', laureate['id'])
    print('Имя:', laureate['name'])
    for prize in laureate['prizes_relevant']:
        print('Год награждения:', prize['award_year'])

# 5. Есть ли повторяющиеся id?
print('\nКоличество повторов id:', duplicates)

# 6. Есть ли разрывы между соседними id?
gaps = find_id_gaps(ids)
print('\nКоличество разрывов:', len(gaps))
print('Пары id с разрывами:', gaps)

Всего записей: 1028

Все ли значения полей заполнены?
Нет

Пропуски у людей:
id: 0 (0.0%)
name: 0 (0.0%)
gender: 0 (0.0%)
birth_year: 0 (0.0%)
country_birth: 3 (0.3%)
country_now: 3 (0.3%)
prizes_relevant: 0 (0.0%)
type_: 0 (0.0%)

Пропуски у организаций:
id: 0 (0.0%)
name: 0 (0.0%)
founded_year: 1 (3.57%)
country_founded: 4 (14.29%)
country_now: 4 (14.29%)
prizes_relevant: 0 (0.0%)
type_: 0 (0.0%)

Минимальный id: 1
Максимальный id: 1070

id: 1
Имя: Wilhelm Conrad Röntgen
Год награждения: 1901

id: 1070
Имя: Michael Woodford
Год награждения: 2026

Количество повторов id: 0

Количество разрывов: 24
Пары id с разрывами: [(6, 8), (93, 95), (169, 172), (255, 257), (468, 470), (503, 505), (520, 523), (541, 543), (569, 571), (590, 592), (594, 596), (597, 600), (610, 613), (615, 617), (631, 633), (635, 637), (645, 647), (655, 657), (783, 786), (787, 789), (858, 861), (896, 903), (949, 958), (1009, 1011)]



Поскольку это курс по Python, а не по статистике или визуализации данных, мы не задаем жесткие рамки того, что вы должны проанализировать.
Однако это отличная возможность представить себя в роли разработчика базы данных или, например, библиотеки pandas.

В коде нужно соблюдать следующие правила:

1. должны быть реализованы  несколько способов работы с данными (среднее/медианное/максимальное значение метрики, количество примеров, вывод топ-N объектов по значению метрики).
2. анализ выбранных вами тем должен включать в себя анализ метрик в тотале, по категориям и в динамике по годам или десятилетиям.
3. проанализировать стоит 2-3 темы. Вы можете использовать темы ниже или взять свои.
2. вывод ноутбука должен быть понятен обывателю. Это может быть визуализация в matplotlib/seaborn. Сопровождайте вывод markdown-ячейками с вашими  мыслями, пояснениями и инсайтами, которые удалось извлечь.
3. Нельзя использовать pandas и numpy, но можно вдохновляться их интерфейсом и функционалом при дизайне ваших функций.


Из пункта 2 следует, что вам придется работать не только с атрибутами лауреатов, но и самих призов.
Информацию о призах нужно доставать и записывать как атрибут уже лауреата. Функции, работающие с призами (например, факт того, что лауреат отказался от какого-то из призов) можно реализовать в ноутбуке, но если вы создаете более общие вспомогательные функции, выносите их в отдельные модули.

Мы рекомендуем следующую структуру кода:

* `dataset_helpers.py` в котором реализованы фильтрация списка словарей (плоских) по условиям (аналог sql-ного where), а также аналог sql-ного групбая. А также функция для применения агрегаций к списку словарей
* `aggregations.py` в котором реализованы функции расчета среднего/медианы/других эвристик и мб вывод топ n-значений по какой-то метрике.
* `atribute_manager.py` в котором реализован переиспользуемый интерфейс для добавления (и мб удаления) полей.

Замечание насчет group-by. 

На игрушесном примере она может работать примерно так:
```
    Входные данные: [
        {'name': 'Alice', 'country': 'USA', 'age': 25},
        {'name': 'Bob', 'country': 'UK', 'age': 30},
        {'name': 'Charlie', 'country': 'USA', 'age': 25}
    ]
    
    group_by_attributes(data, ['country', 'age']) → {
        ('USA', 25): [
            {'name': 'Alice', 'country': 'USA', 'age': 25},
            {'name': 'Charlie', 'country': 'USA', 'age': 25}
        ],
        ('UK', 30): [
            {'name': 'Bob', 'country': 'UK', 'age': 30}
        ]
    }
```

Замечание насчет agregations.py
1. Агрегации соответственно должны применяться к спискам вида [
    {'name': 'Alice', 'country': 'USA', 'age': 25},
    {'name': 'Charlie', 'country': 'USA', 'age': 25}
]
2. Пользуйтесь модулем statistics из стандартной библиотеки питона.


Эта структура одна из возможных для организации вашего кода, главное, чтобы эти достаточно общие функции блыи переиспользованы

----------

При анализе данных вам понадобятся более конкретные функции, например filter_by_num_prizes или output_metric_by_category. Их пишите прямо в ноутбуке и сопровождайте коментами.

Ниже мы приводим некоторые примеры вопросов, которые может быть интересно задать к данным, но если у вас есть желание, можете поисследовать датасет с любой интересной вам стороны!


**2.1. Топ-статистика по странам**
- Определите топ-5 стран по общему количеству лауреатов
- Постройте аналогичные рейтинги отдельно по категориям (физика, химия, медицина и т.д.)

**2.2. Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

**2.3. Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?
- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

**2.4. Миграции лауреатов**
- У скольки лауреатов исходная страна и страна на момент вручения нобелевской премии различаются? Всегда ли это различие объясняется миграцией? Добавьте вспомогательное поле 'country_of_origin_adjusted_for_migration', чтобы различие между этим полем и страной на момент вручения объяснялось миграцией.
- Выявите страны с наибольшей долей "оттекших" и "притекших" лауреатов.
- Есть ли оттоки среди организаций?

**2.5. Многократные лауреаты**
- Найдите лауреатов, получившие больше одного приза. Кто получил больше 2 раз? 
- Совпадает ли категория второго приза с первой?  
- Сколько в среднем времени прошло между получением первой и второй премии?

**2.6 Отказы от премий**
- Определите, в каких категориях чаще отказывались от премий?
- Есть ли отказы среди организаций?

## **Возраст при награждении**
- Рассчитать средний возраст лауреатов на момент получения **первого** приза. Рассмотрите только лауреатов-людей.
- Исследуйте, как средний возраст получения первого приза различается по категориям
- Различаются ли средний и медианный возрасты? 

### Для каждого лауреата найдём год первой премии и рассчитаем возраст на момент награждения.

In [6]:
def get_age_first_prize(person: dict) -> int | None:
    """
    считает возраст лауреата при получении первой премии
    """
    if person['birth_year'] is None:
        return None
    years = []
    for prize in person['prizes_relevant']:
        years.append(prize['award_year'])
    if not years:
        return None
    return min(years) - person['birth_year']

### Соберём возраст всех лауреатов-людей в один список

In [7]:
ages = []
for person in persons:
    age = get_age_first_prize(person)
    if age is not None:
        ages.append(age)

### Посчитаем количество лауреатов, средний и медианный возраст, а также максимальное значение

In [8]:
print(calculate_stats(ages))

{'count': 1000, 'mean': 60.606, 'median': 61.0, 'max': 97}


### Подготовим данные для сравнения возраста по категориям и десятилетиям

In [9]:
first_prizes_data = []

for person in persons:
    first_prize = None
    for prize in person['prizes_relevant']:
        if (
            first_prize is None
            or prize['award_year'] < first_prize['award_year']
        ):
            first_prize = prize
    if first_prize is not None:
        first_prizes_data.append({
            'age': get_age_first_prize(person),
            'category': first_prize['category_en'],
            'award_year': first_prize['award_year'],
            'decade': first_prize['award_year'] // 10 * 10
        })

### Сравним возраст первого награждения в разных категориях премии

In [10]:
groups = group_by_attributes(first_prizes_data, ['category'])

for key, group in groups.items():
    ages = []
    for row in group:
        ages.append(row['age'])
    stats = calculate_stats(ages)

    print(key[0], stats)

Economic Sciences {'count': 101, 'mean': 67.20792079207921, 'median': 67, 'max': 90}
Physics {'count': 230, 'mean': 57.791304347826085, 'median': 56.0, 'max': 96}
Chemistry {'count': 199, 'mean': 59.4321608040201, 'median': 58, 'max': 97}
Literature {'count': 123, 'mean': 65.04065040650407, 'median': 67, 'max': 88}
Peace {'count': 112, 'mean': 60.973214285714285, 'median': 62.0, 'max': 87}
Physiology or Medicine {'count': 235, 'mean': 59.02127659574468, 'median': 58, 'max': 87}


### Посмотрим, как изменялся возраст первого награждения по десятилетиям

In [11]:
print('Динамика по десятилетиям \n')
groups = group_by_attributes(first_prizes_data, ['decade'])
for key, group in sorted(groups.items()):
    ages = []
    for row in group:
        ages.append(row['age'])
    stats = calculate_stats(ages)
    print(
        key[0],
        'Количество:', stats['count'],
        'Средний возраст:', round(stats['mean'], 1),
        'Медиана:', stats['median']
    )

Динамика по десятилетиям 

1900 Количество: 56 Средний возраст: 57.8 Медиана: 58.0
1910 Количество: 37 Средний возраст: 53.1 Медиана: 50
1920 Количество: 54 Средний возраст: 55.0 Медиана: 56.0
1930 Количество: 55 Средний возраст: 51.6 Медиана: 49
1940 Количество: 40 Средний возраст: 58.0 Медиана: 56.0
1950 Количество: 71 Средний возраст: 54.3 Медиана: 54
1960 Количество: 74 Средний возраст: 56.4 Медиана: 57.0
1970 Количество: 102 Средний возраст: 59.3 Медиана: 59.0
1980 Количество: 93 Средний возраст: 60.5 Медиана: 60
1990 Количество: 101 Средний возраст: 62.1 Медиана: 62
2000 Количество: 119 Средний возраст: 65.9 Медиана: 64
2010 Количество: 117 Средний возраст: 66.9 Медиана: 69
2020 Количество: 81 Средний возраст: 69.4 Медиана: 69


### Возраст при получении первой Нобелевской премии

Для расчета возраста использовалась разница между годом первого награждения и годом рождения. Месяц и день рождения не учитывались, поэтому фактический возраст некоторых лауреатов может отличаться на один год.

Средний возраст первого награждения среди 1000 лауреатов составил 60,6 года, медианный 61 год. Показатели практически совпадают, что может говорить об отсутствии выраженной асимметрии распределения. По категориям есть различия: самый высокий средний возраст у лауреатов по экономике (67,2 года), а самый низкий по физике (57,8 года).

Если посмотреть на динамику, видно, что со временем возраст награждения увеличивался. В 1930-х средний возраст составлял 51,6 года, в 1990-х - 62,1 года, а в 2010-х уже 66,9 года.

Одна из возможных причин заключается в том, что между научным открытием и его признанием проходит больше времени.
Также интересно, что в 2000 и 2010 годах в нашей выборке впервые получили премию больше людей, чем в начале XX века: 119 и 117 против 56 в 1900.
При этом 2020-е еще не закончились, поэтому данные за это десятилетие неполные.

В 1910-х и 1940-х годах количество впервые награжденных лауреатов было сравнительно небольшим: 37 и 40 человек. Вероятно, на это повлияли Первая и Вторая мировые войны, поскольку в некоторые военные годы Нобелевские премии не присуждались. При этом нельзя утверждать, что войны повлияли именно на средний возраст лауреатов.

## **Гендерное распределение**
- Как менялось количество женщин-лауреатов по десятилетиям?
- Сравните распределение лауреаток по категориям. Выведите значения суммарно за все годы и по декадам.
- В какой из категорий больше всего женщин лауреаток изначально и в последние пару десятилетий?

### Соберем данные о поле лауреатов и их награждениях

In [12]:
gender_data = []
for person in persons:
    for prize in person['prizes_relevant']:
        gender_data.append({
            'gender': person['gender'],
            'category': prize['category_en'],
            'decade': prize['award_year'] // 10 * 10
        })

### Посчитаем количество и долю награждений женщин в каждой категории

In [13]:
groups = group_by_attributes(gender_data, ['category'])

for key, group in groups.items():
    women = 0
    for row in group:
        if row['gender'] == 'female':
            women += 1
    share = round(women / len(group) * 100, 2)
    print(key[0], 'Женщин:', women, 'Доля:', share, '%')

Economic Sciences Женщин: 3 Доля: 2.97 %
Physics Женщин: 5 Доля: 2.16 %
Chemistry Женщин: 8 Доля: 3.96 %
Literature Женщин: 19 Доля: 15.45 %
Peace Женщин: 21 Доля: 18.58 %
Physiology or Medicine Женщин: 14 Доля: 5.96 %


### Посмотрим, как менялась доля награждений женщин по десятилетиям

In [14]:
groups = group_by_attributes(gender_data, ['decade'])

for key, group in sorted(groups.items()):
    women = 0
    for row in group:
        if row['gender'] == 'female':
            women += 1
    share = round(women / len(group) * 100, 2)
    print(
        key[0],
        'Женщин:', women,
        'Доля:', share, '%'
    )

1900 Женщин: 3 Доля: 5.36 %
1910 Женщин: 1 Доля: 2.63 %
1920 Женщин: 2 Доля: 3.7 %
1930 Женщин: 3 Доля: 5.45 %
1940 Женщин: 3 Доля: 7.5 %
1950 Женщин: 0 Доля: 0.0 %
1960 Женщин: 3 Доля: 4.0 %
1970 Женщин: 4 Доля: 3.88 %
1980 Женщин: 4 Доля: 4.26 %
1990 Женщин: 7 Доля: 6.93 %
2000 Женщин: 11 Доля: 9.24 %
2010 Женщин: 13 Доля: 11.11 %
2020 Женщин: 16 Доля: 19.51 %


### Сравним, в каких категориях женщины получали премии в разные десятилетия.

In [15]:
women_data = []

for row in gender_data:
    if row['gender'] == 'female':
        women_data.append(row)
groups = group_by_attributes(women_data, ['decade', 'category'])
for key, group in sorted(groups.items()):
    print(key[0], key[1], len(group))

1900 Literature 1
1900 Peace 1
1900 Physics 1
1910 Chemistry 1
1920 Literature 2
1930 Chemistry 1
1930 Literature 1
1930 Peace 1
1940 Literature 1
1940 Peace 1
1940 Physiology or Medicine 1
1960 Chemistry 1
1960 Literature 1
1960 Physics 1
1970 Peace 3
1970 Physiology or Medicine 1
1980 Peace 1
1980 Physiology or Medicine 3
1990 Literature 3
1990 Peace 3
1990 Physiology or Medicine 1
2000 Chemistry 1
2000 Economic Sciences 1
2000 Literature 3
2000 Peace 2
2000 Physiology or Medicine 4
2010 Chemistry 1
2010 Economic Sciences 1
2010 Literature 3
2010 Peace 5
2010 Physics 1
2010 Physiology or Medicine 2
2020 Chemistry 3
2020 Economic Sciences 1
2020 Literature 4
2020 Peace 4
2020 Physics 2
2020 Physiology or Medicine 2


### Гендерное распределение лауреатов

Среди 1000 лауреатов 69 женщин и 931 мужчина. Доля женщин составляет всего 6,9%.

Больше всего награждений женщин приходится на премию мира (21) и литературу (19). В этих категориях доля женщин также самая высокая: 18,58% и 15,45%. Меньше всего женщин в физике (2,16%) и экономике (2,97%).

Если посмотреть на динамику, видно, что со временем женщин среди награжденных становится больше. Например, в 1950 не было ни одной женщины, в 1990 их доля составляла 6,93%, а в 2020 уже 19,51%.

Такое распределение может быть связано в том числе с историческими причинами. Раньше женщинам было намного сложнее получить образование и заниматься наукой. Также известны случаи, когда Нобелевскую премию за открытие жещины, получал мужчина.
Получается, что низкая доля женщин среди лауреатов не обязательно означает, что женщины делали меньше открытий. На результаты могли влиять ограничения в образовании и признании научных достижений. При этом в последние десятилетия ситуация заметно меняется.

В начале XX века женщины получали премии редко и в отдельных категориях. В 2010 больше всего награждений женщин было в категории премии мира (5), а в 2020 лидируют литература и премия мира (по 4). При этом в 2020 женщины представлены уже во всех шести категориях.

Также можно заметить, что количество женщин - 69, а награждений 70, значит одна женщина получила премию дважды.

Таким образом, доля женщин среди награжденных постепенно растет, хотя между категориями все еще сохраняются заметные различия.

## Доп. верхнеуровневый анализ

### Дополнительно посмотрим общее количество награждений, их статусы и самые распространённые категории

In [16]:
prizes_data = []

for laureate in laureates_data:
    for prize in laureate['prizes_relevant']:
        prizes_data.append(prize)
print('Всего награждений:', len(prizes_data))

# статусы премий
statuses = group_by_attributes(prizes_data, ['prize_status'])

print('\nСтатусы премий:')
for key, group in statuses.items():
    print(key[0], len(group))

# топ 5
groups = group_by_attributes(prizes_data, ['category_en'])
category_counts = []
for key, group in groups.items():
    category_counts.append((len(group), key[0]))
print('\nТоп-5 категорий:')
for count, category in sorted(category_counts, reverse=True)[:5]:
    print(category, count)

Всего награждений: 1036

Статусы премий:
received 1033
restricted 1
declined 2

Топ-5 категорий:
Physiology or Medicine 235
Physics 231
Chemistry 202
Peace 144
Literature 123


### Дополнительный анализ премий

В датасете 1028 лауреатов и 1036 записей о премиях. Разница связана с тем, что некоторые лауреаты получали премию несколько раз.

Практически все премии имеют статус `received` (1033 записи). При этом встретилось два отказа от премии (`declined`) и одна запись со статусом `restricted`.

Больше всего награждений приходится на медицину (235), физику (231) и химию (202). При этом мы считаем записи о награждениях лауреатов, поэтому одна премия, разделённая между несколькими людьми, учитывается несколько раз.

## Итоги

В работе удалось обработать JSON-файл с данными о Нобелевских лауреатах и провести небольшой исследовательский анализ.

Основные результаты:

1. В датасете 1028 лауреатов, большинство полей заполнено, повторяющихся ID нет.
2. Средний возраст получения первой премии составляет 60,6 года. При этом в последние десятилетия лауреаты получают первую премию в более старшем возрасте.
3. Женщины составляют 6,9% среди лауреатов-людей. Их доля среди награждённых со временем увеличивается, особенно в последние десятилетия.
4. Больше всего награждений приходится на медицину, физику и химию. Практически все премии имеют статус `received`.

Для обработки данных и расчёта статистик использовались отдельные функции, которые можно повторно применять к разным частям датасета.

---------

## Бонусное задание на +2 балла


### Лингвистический анализ мотиваций

**5.1. Анализ текстов мотиваций**
- Соберите корпус текстов мотиваций вручения премий. Для этого измените конфиг обработки призов.
- Выделите самые частые топ-50 слов. Какие из этих слов самые частые в принципе в английском языке (такие, как for, the и другие), а какие кажутся важными именно в контексте вручения Нобелевской премии (например, 'chemistry')?
- Самые частые слова, которые также являются частыми во всем английском языке занесите в отдельный конфиг-файл под названием `stopwords.py`. Вычистите их из текстов. Какие слова являются самыми частыми теперь? Сравните множества топ-10 самых частых слов по разным категориям


- Если есть желание, посчитайте TF-idF + сравнените топ слов по этой метрике с просто самыми популярными словами)

In [17]:
motivations = []
for laureate in laureates_data:
    for prize in laureate['prizes_relevant']:
        if prize['motivation_en']:
            motivations.append({
                'category': prize['category_en'],
                'text': prize['motivation_en']
            })

words = []
for row in motivations:
    words.extend(re.findall(r'[a-z]+', row['text'].lower()))
    
print('Количество текстов:', len(motivations))
print('\nТоп-50 слов:')

for word, count in Counter(words).most_common(50):
    print(word, count)

Количество текстов: 1036

Топ-50 слов:
of 1291
the 1248
for 1149
and 670
in 449
their 394
his 376
to 278
discovery 211
discoveries 172
a 151
concerning 124
development 110
work 106
on 101
with 99
which 99
by 82
theory 74
contributions 61
has 60
structure 58
that 55
peace 53
recognition 48
economic 45
human 44
chemical 43
as 42
analysis 42
new 41
its 39
synthesis 38
pioneering 38
an 38
fundamental 37
efforts 36
between 34
chemistry 34
he 34
her 32
studies 31
through 31
international 30
method 30
especially 29
methods 29
having 28
role 27
research 26


### В топ-50 оказалось много обычных английских слов, например `the`, `of`, `for`. Они часто встречаются в текстах, но мало говорят о причинах награждений. Уберём их и посмотрим, какие слова теперь самые популярные.

In [18]:
clean_words = []
for word in words:
    if word not in STOPWORDS:
        clean_words.append(word)

print('Топ-20 слов после очистки:')

for word, count in Counter(clean_words).most_common(20):
    print(word, count)

Топ-20 слов после очистки:
discovery 211
discoveries 172
concerning 124
development 110
work 106
theory 74
contributions 61
structure 58
peace 53
recognition 48
economic 45
human 44
chemical 43
analysis 42
new 41
synthesis 38
pioneering 38
fundamental 37
efforts 36
between 34


### Теперь найдём по 10 самых частых слов для каждой категории Нобелевской премии и сравним, какие слова встречаются в нескольких категориях.

In [19]:
groups = group_by_attributes(motivations, ['category'])
top10 = {}
for key, group in groups.items():
    category = key[0]
    category_words = []
    for row in group:
        row_words = re.findall(r'[a-z]+', row['text'].lower())
        for word in row_words:
            if word not in STOPWORDS:
                category_words.append(word)
    popular = Counter(category_words).most_common(10)
    top10[category] = set(word for word, count in popular)

    print('\n', category)
    print(popular)


 Economic Sciences
[('theory', 43), ('economic', 41), ('analysis', 35), ('contributions', 18), ('having', 18), ('pioneering', 15), ('research', 13), ('markets', 10), ('financial', 8), ('empirical', 8)]

 Physics
[('discovery', 100), ('development', 30), ('discoveries', 27), ('theory', 25), ('contributions', 23), ('quantum', 22), ('fundamental', 21), ('structure', 20), ('work', 20), ('physics', 19)]

 Chemistry
[('development', 52), ('discovery', 36), ('chemistry', 33), ('chemical', 31), ('synthesis', 28), ('work', 26), ('structure', 23), ('organic', 23), ('studies', 20), ('contributions', 17)]

 Literature
[('human', 19), ('life', 19), ('poetry', 18), ('poetic', 17), ('who', 16), ('art', 16), ('great', 12), ('power', 11), ('narrative', 11), ('recognition', 10)]

 Peace
[('peace', 53), ('efforts', 36), ('work', 32), ('international', 26), ('rights', 23), ('between', 19), ('war', 19), ('peaceful', 17), ('struggle', 16), ('human', 16)]

 Physiology or Medicine
[('discoveries', 138), ('co

In [20]:
for category1 in top10:
    for category2 in top10:
        if category1 < category2:
            common = top10[category1] & top10[category2]
            print(category1, '/', category2, ':', common)

Economic Sciences / Physics : {'theory', 'contributions'}
Economic Sciences / Literature : set()
Economic Sciences / Peace : set()
Economic Sciences / Physiology or Medicine : set()
Physics / Physiology or Medicine : {'discoveries', 'discovery', 'development', 'work'}
Chemistry / Economic Sciences : {'contributions'}
Chemistry / Physics : {'contributions', 'development', 'structure', 'discovery', 'work'}
Chemistry / Literature : set()
Chemistry / Peace : {'work'}
Chemistry / Physiology or Medicine : {'discovery', 'development', 'work'}
Literature / Physics : set()
Literature / Peace : {'human'}
Literature / Physiology or Medicine : set()
Peace / Physics : {'work'}
Peace / Physiology or Medicine : {'work'}


### Вывод

До удаления стоп-слов самыми частыми оказались обычные английские слова, такие как **of, the, for, and**. После очистки на первые места вышли **discovery, discoveries, concerning, development, work**. При этом слово **concerning** тоже не несёт особого смысла, поскольку часто используется в формулировках мотиваций.

По категориям заметны различия. Например, для физики характерны слова **discovery, quantum, theory**, для экономики **economic, analysis, theory**, для литературы **poetry, life, poetic**, а для премии мира **peace, efforts, international**.

При сравнении топ-10 оказалось, что у физики и химии совпадают 5 слов, а у физики и медицины 4 слова. При этом у литературы нет общих слов из топ-10 с физикой и химией.

Получается, что после удаления стоп-слов тексты мотиваций лучше отражают особенности каждой категории. При этом у научных категорий достаточно много общих слов, связанных с открытиями и исследованиями.